# YouTube Channel Automation - @clarityinthequran

**Runs entirely in Colab. Nothing local. iPhone-only workflow.**

| Cell | What it does |
|---|---|
| 1 | Installs libraries, mounts Drive, authenticates YouTube |
| 2 | Upload video / read comments / reply to comments |
| 3 | Backs up your channel to Drive with `yt-dlp` |

**Before you run anything**, finish the Google Cloud setup in
`youtube-automation/README.md` and drop `client_secret.json` into your
private Drive secrets folder.

Run the cells **in order**. Cell 1 must run first, every session.

In [ ]:
# ============================================================
#  CELL 1 -- ENVIRONMENT, DRIVE MOUNT, YOUTUBE AUTH
#  Run this first, every single session.
# ============================================================

# ------------------------------------------------------------
#  1A. THE ONLY LINES YOU EDIT
# ------------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive"

# Where your CapCut exports land (the upload queue).
UPLOAD_FOLDER_PATH = DRIVE_ROOT + "/YOUR_UPLOAD_FOLDER"

# Where channel backups get written.
BACKUP_FOLDER_PATH = DRIVE_ROOT + "/YOUR_BACKUP_FOLDER"

# Private folder holding client_secret.json (token.json is
# written here too). Do NOT share this folder with anyone.
SECRETS_FOLDER_PATH = DRIVE_ROOT + "/YOUR_SECRETS_FOLDER"

# Your channel handle, including the @.
YOUR_CHANNEL_HANDLE = "@clarityinthequran"


# ------------------------------------------------------------
#  1B. INSTALL LIBRARIES  (about 40 seconds)
# ------------------------------------------------------------
!pip install -q --upgrade \
    google-api-python-client \
    google-auth-oauthlib \
    google-auth-httplib2 \
    yt-dlp


# ------------------------------------------------------------
#  1C. MOUNT DRIVE + NATIVE COLAB SIGN-IN
# ------------------------------------------------------------
import os

from google.colab import drive
from google.colab import auth

# Mounts your Drive at /content/drive. Tap the popup, approve.
drive.mount("/content/drive")

# Signs this Colab VM in as you. Covers Drive and Cloud APIs.
# NOTE: this does NOT grant YouTube scopes -- Colab's built-in
# auth cannot request youtube.upload. That is what 1D is for.
auth.authenticate_user()

# Create the three folders if they do not exist yet, so no
# later cell can crash on a missing path.
for FOLDER in (UPLOAD_FOLDER_PATH,
               BACKUP_FOLDER_PATH,
               SECRETS_FOLDER_PATH):
    os.makedirs(FOLDER, exist_ok=True)
    print("ready ->", FOLDER)


# ------------------------------------------------------------
#  1D. YOUTUBE OAUTH -- "paste the link" flow, no local server
# ------------------------------------------------------------
import json
from urllib.parse import urlparse, parse_qs

from google_auth_oauthlib.flow import Flow
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# upload  = post new videos
# force-ssl = read comments, post replies, edit your videos
YOUTUBE_SCOPES = [
    "https://www.googleapis.com/auth/youtube.upload",
    "https://www.googleapis.com/auth/youtube.force-ssl",
]

CLIENT_SECRET_FILE = SECRETS_FOLDER_PATH + "/client_secret.json"
TOKEN_FILE = SECRETS_FOLDER_PATH + "/token.json"

# Google killed the old copy/paste ("oob") flow in Jan 2023, so
# we send the redirect to localhost instead. Your iPhone cannot
# open localhost -- the page WILL fail. That is expected: the
# code we need is sitting in the failed page's URL.
REDIRECT_URI = "http://localhost:8080/"

# localhost is plain http, and Google sometimes returns extra
# scopes. These two flags stop oauthlib refusing on both counts.
os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"
os.environ["OAUTHLIB_RELAX_TOKEN_SCOPE"] = "1"


def get_youtube_service():
    """Return an authenticated YouTube Data API v3 client."""
    CREDS = None

    # Step 1: reuse the saved token from a previous session.
    if os.path.exists(TOKEN_FILE):
        CREDS = Credentials.from_authorized_user_file(
            TOKEN_FILE, YOUTUBE_SCOPES)

    # Step 2: token expired? Refresh it silently, no tapping.
    if CREDS and CREDS.expired and CREDS.refresh_token:
        try:
            CREDS.refresh(Request())
            print("Token refreshed silently.")
        except Exception as ERROR:
            print("Refresh failed (", ERROR, ") -- re-authorising.")
            CREDS = None

    # Step 3: first run, or the refresh token died. Paste flow.
    if not CREDS or not CREDS.valid:
        if not os.path.exists(CLIENT_SECRET_FILE):
            raise FileNotFoundError(
                "client_secret.json missing at " + CLIENT_SECRET_FILE)

        FLOW = Flow.from_client_secrets_file(
            CLIENT_SECRET_FILE,
            scopes=YOUTUBE_SCOPES,
            redirect_uri=REDIRECT_URI,
        )
        AUTH_URL, _ = FLOW.authorization_url(
            access_type="offline",   # ask for a refresh token
            prompt="consent",        # force one to be issued
            include_granted_scopes="true",
        )

        print("")
        print("=" * 54)
        print(" 1. Long-press the link below, open in a new tab.")
        print(" 2. Pick the Google account that owns the channel.")
        print(" 3. On 'Google hasn't verified this app', tap")
        print("    Advanced -> Go to <your app> (unsafe).")
        print(" 4. Approve both YouTube permissions.")
        print(" 5. Safari shows 'cannot connect to the server'.")
        print("    THIS IS CORRECT. Tap the address bar, copy the")
        print("    ENTIRE url, come back here and paste it.")
        print("=" * 54)
        print("")
        print(AUTH_URL)
        print("")

        PASTED = input("Paste the localhost URL here: ").strip()

        # Accept a full URL or just the bare code, either works.
        if PASTED.startswith("http"):
            QUERY = parse_qs(urlparse(PASTED).query)
            CODE = QUERY.get("code", [""])[0]
        else:
            CODE = PASTED

        if not CODE:
            raise RuntimeError("No ?code= found in what you pasted.")

        FLOW.fetch_token(code=CODE)
        CREDS = FLOW.credentials

        # Save to Drive so you never repeat this (see README on
        # the 7-day expiry if your app is still in "Testing").
        with open(TOKEN_FILE, "w") as HANDLE:
            HANDLE.write(CREDS.to_json())
        print("")
        print("Token saved ->", TOKEN_FILE)

    return build("youtube", "v3", credentials=CREDS)


# Build the client, then prove it is on the right channel.
YOUTUBE = get_youtube_service()

ME = YOUTUBE.channels().list(
    part="snippet,contentDetails,statistics", mine=True).execute()

MY_CHANNEL = ME["items"][0]
MY_CHANNEL_ID = MY_CHANNEL["id"]
MY_PLAYLISTS = MY_CHANNEL["contentDetails"]["relatedPlaylists"]
MY_UPLOADS_PLAYLIST = MY_PLAYLISTS["uploads"]

print("")
print("Connected to :", MY_CHANNEL["snippet"]["title"])
print("Channel ID   :", MY_CHANNEL_ID)
print("Uploads list :", MY_UPLOADS_PLAYLIST)
print("Video count  :", MY_CHANNEL["statistics"].get("videoCount"))

## Cell 2 - Upload, read comments, reply

Defines the functions only; running it posts nothing. Usage examples are
at the bottom of the cell, commented out.

**Quota:** you get 10,000 units/day. An upload costs 1,600 (so ~6 uploads
per day), a reply costs 50, reading comments costs 1 per page.

In [ ]:
# ============================================================
#  CELL 2 -- YOUTUBE ACTIONS: UPLOAD / READ / REPLY
#  Defines functions. Running this cell posts nothing.
# ============================================================

import os
import time
import random
import shutil

from googleapiclient.http import MediaFileUpload
from googleapiclient.errors import HttpError

# Daily quota is 10,000 units. Rough costs per call:
#   upload_video ......... 1,600 units  (about 6 uploads/day)
#   read_latest_comments ...... 1 unit  per page of 100
#   reply_to_comment ......... 50 units
#   set_video_privacy ........ 50 units


# ------------------------------------------------------------
#  2A. SEE WHAT IS SITTING IN YOUR DRIVE UPLOAD FOLDER
# ------------------------------------------------------------
def list_drive_uploads():
    """Print every video file waiting in UPLOAD_FOLDER_PATH."""
    VIDEO_TYPES = (".mp4", ".mov", ".m4v", ".webm", ".mkv", ".avi")
    FOUND = sorted(
        NAME for NAME in os.listdir(UPLOAD_FOLDER_PATH)
        if NAME.lower().endswith(VIDEO_TYPES)
    )
    if not FOUND:
        print("No videos in", UPLOAD_FOLDER_PATH)
    for NAME in FOUND:
        SIZE_MB = os.path.getsize(
            os.path.join(UPLOAD_FOLDER_PATH, NAME)) / (1024 * 1024)
        print("%8.1f MB  %s" % (SIZE_MB, NAME))
    return FOUND


# ------------------------------------------------------------
#  2B. UPLOAD ONE VIDEO FROM DRIVE TO YOUTUBE
# ------------------------------------------------------------
def upload_video(VIDEO_FILE_NAME,
                 VIDEO_TITLE,
                 VIDEO_DESCRIPTION="",
                 VIDEO_TAGS=None,
                 PRIVACY_STATUS="private",
                 CATEGORY_ID="22",
                 MADE_FOR_KIDS=False,
                 PUBLISH_AT=None,
                 COPY_TO_LOCAL_FIRST=True,
                 DELETE_FROM_DRIVE_AFTER=False):
    """
    Upload a file from UPLOAD_FOLDER_PATH to YouTube.

    VIDEO_FILE_NAME : file name only, e.g. "kahf_reminder.mp4"
    PRIVACY_STATUS  : "private" | "unlisted" | "public"
    CATEGORY_ID     : "22" People & Blogs, "27" Education,
                      "24" Entertainment, "25" News & Politics
    PUBLISH_AT      : ISO-8601 UTC, e.g. "2026-09-01T14:00:00Z".
                      Only honoured while PRIVACY_STATUS is
                      "private" -- YouTube flips it public then.

    Returns the new video id.
    """
    SOURCE_PATH = os.path.join(UPLOAD_FOLDER_PATH, VIDEO_FILE_NAME)
    if not os.path.exists(SOURCE_PATH):
        raise FileNotFoundError(
            "Not in your Drive folder: " + SOURCE_PATH)

    # Drive's FUSE mount stalls on long resumable uploads, so we
    # copy to the VM's own disk first. Far more reliable.
    if COPY_TO_LOCAL_FIRST:
        LOCAL_PATH = "/content/" + VIDEO_FILE_NAME
        print("Copying Drive -> local disk ...")
        shutil.copy(SOURCE_PATH, LOCAL_PATH)
    else:
        LOCAL_PATH = SOURCE_PATH

    SIZE_MB = os.path.getsize(LOCAL_PATH) / (1024 * 1024)
    print("Ready to upload: %.1f MB" % SIZE_MB)

    REQUEST_BODY = {
        "snippet": {
            "title": VIDEO_TITLE,
            "description": VIDEO_DESCRIPTION,
            "tags": VIDEO_TAGS or [],
            "categoryId": CATEGORY_ID,
        },
        "status": {
            "privacyStatus": PRIVACY_STATUS,
            "selfDeclaredMadeForKids": MADE_FOR_KIDS,
        },
    }
    if PUBLISH_AT:
        REQUEST_BODY["status"]["publishAt"] = PUBLISH_AT

    # resumable=True survives dropped chunks on mobile tethering.
    MEDIA = MediaFileUpload(LOCAL_PATH,
                            chunksize=5 * 1024 * 1024,
                            resumable=True,
                            mimetype="video/*")

    REQUEST = YOUTUBE.videos().insert(part="snippet,status",
                                      body=REQUEST_BODY,
                                      media_body=MEDIA)

    RESPONSE = None
    RETRIES = 0
    while RESPONSE is None:
        try:
            STATUS, RESPONSE = REQUEST.next_chunk()
            if STATUS:
                PERCENT = int(STATUS.progress() * 100)
                print("uploading ... %d%%" % PERCENT)
        except HttpError as ERROR:
            # 5xx / 429 are transient. Anything else is a real bug.
            TRANSIENT = (429, 500, 502, 503, 504)
            if ERROR.resp.status in TRANSIENT and RETRIES < 5:
                RETRIES += 1
                WAIT = (2 ** RETRIES) + random.random()
                print("transient error, retry %d in %.1fs" %
                      (RETRIES, WAIT))
                time.sleep(WAIT)
            else:
                raise

    VIDEO_ID = RESPONSE["id"]
    print("")
    print("DONE. Video id:", VIDEO_ID)
    print("Watch : https://youtu.be/" + VIDEO_ID)
    print("Studio: https://studio.youtube.com/video/"
          + VIDEO_ID + "/edit")
    print("")
    print("NOTE: until your Cloud project passes YouTube's API")
    print("compliance audit, this upload is LOCKED PRIVATE. Set it")
    print("public from the YouTube Studio app. See README.")

    # Free the VM's disk, and optionally clear the Drive queue.
    if COPY_TO_LOCAL_FIRST:
        os.remove(LOCAL_PATH)
    if DELETE_FROM_DRIVE_AFTER:
        os.remove(SOURCE_PATH)
        print("Removed from Drive queue:", VIDEO_FILE_NAME)

    return VIDEO_ID


# ------------------------------------------------------------
#  2C. READ THE LATEST COMMENTS
# ------------------------------------------------------------
def read_latest_comments(HOW_MANY=20, VIDEO_ID=None, SHOW=True):
    """
    Newest comment threads, channel-wide or for one video.

    HOW_MANY : how many threads to pull back
    VIDEO_ID : leave None for every video on the channel

    Returns a list of dicts. The "comment_id" field is what you
    hand to reply_to_comment().
    """
    COLLECTED = []
    PAGE_TOKEN = None

    while len(COLLECTED) < HOW_MANY:
        PARAMS = {
            "part": "snippet,replies",
            "maxResults": min(100, HOW_MANY - len(COLLECTED)),
            "order": "time",
            "textFormat": "plainText",
            "pageToken": PAGE_TOKEN,
        }
        # One video, or every video you own.
        if VIDEO_ID:
            PARAMS["videoId"] = VIDEO_ID
        else:
            PARAMS["allThreadsRelatedToChannelId"] = MY_CHANNEL_ID

        RESULT = YOUTUBE.commentThreads().list(**PARAMS).execute()

        for THREAD in RESULT.get("items", []):
            TOP = THREAD["snippet"]["topLevelComment"]
            SNIP = TOP["snippet"]
            COLLECTED.append({
                "comment_id": TOP["id"],          # reply target
                "video_id": SNIP.get("videoId"),
                "author": SNIP["authorDisplayName"],
                "text": SNIP["textDisplay"],
                "likes": SNIP["likeCount"],
                "published": SNIP["publishedAt"],
                "reply_count": THREAD["snippet"]["totalReplyCount"],
            })

        PAGE_TOKEN = RESULT.get("nextPageToken")
        if not PAGE_TOKEN:
            break

    if SHOW:
        for INDEX, ITEM in enumerate(COLLECTED, 1):
            print("-" * 44)
            print("%d. %s   (%d likes, %d replies)" %
                  (INDEX, ITEM["author"], ITEM["likes"],
                   ITEM["reply_count"]))
            print(ITEM["text"])
            print("id:", ITEM["comment_id"])
        print("-" * 44)
        print("total:", len(COLLECTED))

    return COLLECTED


def read_unanswered_comments(HOW_MANY=50):
    """Only the threads you have not replied to yet."""
    ALL_THREADS = read_latest_comments(HOW_MANY, SHOW=False)
    UNANSWERED = [T for T in ALL_THREADS if T["reply_count"] == 0]
    for ITEM in UNANSWERED:
        print("-" * 44)
        print(ITEM["author"], "->", ITEM["text"])
        print("id:", ITEM["comment_id"])
    print("-" * 44)
    print("unanswered:", len(UNANSWERED), "of", len(ALL_THREADS))
    return UNANSWERED


# ------------------------------------------------------------
#  2D. REPLY TO A COMMENT
# ------------------------------------------------------------
def reply_to_comment(COMMENT_ID, REPLY_TEXT):
    """
    Post a reply under a top-level comment.

    COMMENT_ID must be the top-level comment id -- the "id" that
    the two functions above print. YouTube has no nested replies:
    replying to a reply still attaches to the parent thread.
    """
    RESPONSE = YOUTUBE.comments().insert(
        part="snippet",
        body={
            "snippet": {
                "parentId": COMMENT_ID,
                "textOriginal": REPLY_TEXT,
            }
        },
    ).execute()

    print("Replied. Reply id:", RESPONSE["id"])
    return RESPONSE["id"]


# ------------------------------------------------------------
#  2E. CHANGE A VIDEO'S PRIVACY (e.g. private -> public)
# ------------------------------------------------------------
def set_video_privacy(VIDEO_ID, NEW_PRIVACY="public"):
    """
    Flip privacy without wiping the rest of the status block.
    We read the current status first, then patch one field --
    videos.update replaces every field you send.
    """
    CURRENT = YOUTUBE.videos().list(
        part="status", id=VIDEO_ID).execute()
    if not CURRENT["items"]:
        raise ValueError("No video with id " + VIDEO_ID)

    STATUS_BLOCK = CURRENT["items"][0]["status"]
    STATUS_BLOCK["privacyStatus"] = NEW_PRIVACY

    RESPONSE = YOUTUBE.videos().update(
        part="status",
        body={"id": VIDEO_ID, "status": STATUS_BLOCK},
    ).execute()

    print(VIDEO_ID, "is now", RESPONSE["status"]["privacyStatus"])
    return RESPONSE


# ------------------------------------------------------------
#  HOW TO USE -- delete the # to run one
# ------------------------------------------------------------

# list_drive_uploads()

# upload_video(
#     VIDEO_FILE_NAME="YOUR_VIDEO.mp4",
#     VIDEO_TITLE="YOUR TITLE HERE",
#     VIDEO_DESCRIPTION="YOUR DESCRIPTION HERE",
#     VIDEO_TAGS=["quran", "tafsir", "islam"],
#     PRIVACY_STATUS="private",
#     CATEGORY_ID="27",
# )

# read_latest_comments(HOW_MANY=15)

# read_unanswered_comments(HOW_MANY=50)

# reply_to_comment("PASTE_COMMENT_ID_HERE", "Jazak Allahu khayran!")

# set_video_privacy("PASTE_VIDEO_ID_HERE", "public")

## Cell 3 - Channel backup to Drive

The Data API has no download endpoint, so this uses `yt-dlp`. It reads
your **uploads playlist** (from Cell 1), which includes Shorts — the
`/videos` tab does not.

Backups are **incremental**: every finished video id is written to
`_backup_archive.txt` in your backup folder, so re-running only fetches
what is new. Delete that file to force a full re-download.

In [ ]:
# ============================================================
#  CELL 3 -- CHANNEL BACKUP TO DRIVE (yt-dlp)
#  The Data API has no download endpoint, so we use yt-dlp.
# ============================================================

import os
import shutil

from yt_dlp import YoutubeDL

# ------------------------------------------------------------
#  3A. BACKUP SETTINGS
# ------------------------------------------------------------

# Best mp4 up to 1080p. Use height<=2160 for 4K masters.
BACKUP_QUALITY = "bv*[height<=1080]+ba/b[height<=1080]/b"

# Videos per run. Keep it small -- Colab disconnects on idle.
MAX_VIDEOS_PER_RUN = 10

# Optional. If YouTube throws a bot-check, drop a Netscape-format
# cookies.txt here. Also required for PRIVATE videos.
COOKIES_FILE_PATH = SECRETS_FOLDER_PATH + "/cookies.txt"

# The uploads playlist covers Shorts too; the /videos tab does not.
try:
    CHANNEL_SOURCE_URL = ("https://www.youtube.com/playlist?list="
                          + MY_UPLOADS_PLAYLIST)
except NameError:
    CHANNEL_SOURCE_URL = ("https://www.youtube.com/"
                          + YOUR_CHANNEL_HANDLE + "/videos")

# Ledger of what is already saved, so re-runs stay incremental.
ARCHIVE_FILE = BACKUP_FOLDER_PATH + "/_backup_archive.txt"

# We download here first, then move to Drive. Writing straight to
# the Drive mount corrupts yt-dlp's merge step on large files.
LOCAL_WORK_DIR = "/content/yt_backup_tmp"


# ------------------------------------------------------------
#  3B. yt-dlp OPTIONS
# ------------------------------------------------------------
def build_download_options():
    """Options dict for one video download."""
    # %(title).80B truncates by BYTES, so Arabic titles cannot
    # blow past the 255-byte filename limit.
    OPTIONS = {
        "format": BACKUP_QUALITY,
        "merge_output_format": "mp4",
        "outtmpl": (LOCAL_WORK_DIR
                    + "/%(upload_date)s_%(title).80B [%(id)s].%(ext)s"),
        "writeinfojson": True,      # title, description, tags, stats
        "writethumbnail": True,     # the thumbnail image
        "writesubtitles": True,     # your uploaded captions
        "subtitleslangs": ["en.*", "ar.*"],
        "ignoreerrors": True,
        "retries": 5,
        "fragment_retries": 10,
        "concurrent_fragment_downloads": 4,
        "quiet": True,
        "no_warnings": True,
    }

    # Only attach cookies if you actually put a file there.
    if os.path.exists(COOKIES_FILE_PATH):
        OPTIONS["cookiefile"] = COOKIES_FILE_PATH
        print("using cookies from", COOKIES_FILE_PATH)

    # If you hit "Sign in to confirm you're not a bot", uncomment:
    # OPTIONS["extractor_args"] = {"youtube": {"player_client": ["tv"]}}

    return OPTIONS


# ------------------------------------------------------------
#  3C. LIST EVERY VIDEO ON THE CHANNEL (no downloading)
# ------------------------------------------------------------
def list_channel_videos():
    """Return [{'id':..., 'title':...}, ...] for the whole channel."""
    FLAT_OPTIONS = {
        "extract_flat": "in_playlist",
        "ignoreerrors": True,
        "quiet": True,
        "no_warnings": True,
    }
    if os.path.exists(COOKIES_FILE_PATH):
        FLAT_OPTIONS["cookiefile"] = COOKIES_FILE_PATH

    with YoutubeDL(FLAT_OPTIONS) as YDL:
        INFO = YDL.extract_info(CHANNEL_SOURCE_URL, download=False)

    ENTRIES = [ITEM for ITEM in (INFO.get("entries") or []) if ITEM]
    return [{"id": ITEM["id"], "title": ITEM.get("title", "")}
            for ITEM in ENTRIES]


# ------------------------------------------------------------
#  3D. THE ARCHIVE LEDGER
# ------------------------------------------------------------
def load_backed_up_ids():
    """Video ids already saved to Drive."""
    if not os.path.exists(ARCHIVE_FILE):
        return set()
    with open(ARCHIVE_FILE) as HANDLE:
        return {LINE.strip() for LINE in HANDLE if LINE.strip()}


def record_backed_up_id(VIDEO_ID):
    """Mark one id as done -- only ever called after a good move."""
    with open(ARCHIVE_FILE, "a") as HANDLE:
        HANDLE.write(VIDEO_ID + "\n")


# ------------------------------------------------------------
#  3E. RUN THE BACKUP
# ------------------------------------------------------------
def backup_channel(HOW_MANY=MAX_VIDEOS_PER_RUN):
    """
    Download videos not yet backed up, straight into
    BACKUP_FOLDER_PATH on your Drive. Safe to re-run: it skips
    anything already listed in the archive ledger.
    """
    os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
    os.makedirs(BACKUP_FOLDER_PATH, exist_ok=True)

    ALREADY_DONE = load_backed_up_ids()
    ALL_VIDEOS = list_channel_videos()
    PENDING = [V for V in ALL_VIDEOS
               if V["id"] not in ALREADY_DONE]
    TODO = PENDING[:HOW_MANY]

    print("on channel :", len(ALL_VIDEOS))
    print("backed up  :", len(ALREADY_DONE))
    print("this run   :", len(TODO))

    for NUMBER, VIDEO in enumerate(TODO, 1):
        print("")
        print("[%d/%d] %s" % (NUMBER, len(TODO), VIDEO["title"][:55]))

        try:
            with YoutubeDL(build_download_options()) as YDL:
                YDL.download(
                    ["https://www.youtube.com/watch?v="
                     + VIDEO["id"]])
        except Exception as ERROR:
            print("  FAILED:", ERROR)
            continue

        # Move everything this video produced onto Drive, then --
        # and only then -- record it as backed up.
        MOVED = 0
        for NAME in os.listdir(LOCAL_WORK_DIR):
            shutil.move(os.path.join(LOCAL_WORK_DIR, NAME),
                        os.path.join(BACKUP_FOLDER_PATH, NAME))
            MOVED += 1

        if MOVED:
            record_backed_up_id(VIDEO["id"])
            print("  saved %d file(s) to Drive" % MOVED)
        else:
            print("  nothing produced -- leaving it unmarked")

    print("")
    print("Backup run finished ->", BACKUP_FOLDER_PATH)


def backup_single_video(VIDEO_ID_OR_URL):
    """Grab one specific video, ignoring the ledger."""
    os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
    URL = VIDEO_ID_OR_URL
    if not URL.startswith("http"):
        URL = "https://www.youtube.com/watch?v=" + URL

    with YoutubeDL(build_download_options()) as YDL:
        YDL.download([URL])

    for NAME in os.listdir(LOCAL_WORK_DIR):
        shutil.move(os.path.join(LOCAL_WORK_DIR, NAME),
                    os.path.join(BACKUP_FOLDER_PATH, NAME))
    print("saved ->", BACKUP_FOLDER_PATH)


# ------------------------------------------------------------
#  HOW TO USE -- delete the # to run one
# ------------------------------------------------------------

# list_channel_videos()

# backup_channel(HOW_MANY=5)

# backup_single_video("PASTE_VIDEO_ID_HERE")